<a href="https://colab.research.google.com/github/priyanshu-sh04/gemini-python-notebooks/blob/main/Gemini_with_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-generativeai


In [ ]:
import google.genai as genai
import pathlib
import textwrap
from IPython.display import display
from IPython.display import Markdown

In [ ]:
def to_markdown(text):
    text = text.replace('●', ' *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

# Example
input_text = "This is a ● sample text with bullets points."
result = to_markdown(input_text)
display(result)

In [ ]:
from google.colab import userdata


In [ ]:
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
for models in client.models.list():
  print(models.name)
  print(" Display name:", models.display_name)
  print(" Description:", models.description)
  print(" Supported actions:", models.supported_actions)


In [ ]:
for models in client.models.list():
  if "generateContent" in models.supported_actions or []:
    print(models.display_name)

In [ ]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="What is the future of AI"
)


In [ ]:
print(response.text)

In [ ]:
to_markdown(response.text)

In [ ]:
%%time
response2 = client.models.generate_content_stream(
    model="gemini-3-flash-preview",
    contents="What is the future of AI"
)


In [ ]:

for chunk in response2:
    print(chunk.text, end="")


In [ ]:
!curl -o image.jpeg https://images.unsplash.com/photo-1790747754584-15508864e933?q=80&w=1170&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D
!curl -o image2.jpeg https://images.unsplash.com/photo-1790302798119-49a36639e529?q=80&w=1170&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D

In [ ]:
import PIL.Image
img = PIL.Image.open('image.jpeg')
img

Multi-turn Conversation

In [ ]:
from google.genai import types

config = types.GenerateContentConfig(
    system_instruction="Answer in simple Hinglish",
    temperature=0.5,
    top_p=0.9,
    top_k=40,
    max_output_tokens=500,
    stop_sequences=["END"],
    seed=42,
)

In [ ]:
interaction1 = client.interactions.create(
    model="gemini-3.8-flash",
    input="I have two dogs in my house",
    system_instruction="You are a helpful assistant.",
    generation_config={"temperature": 0.5},
    )

print("Response 1: ", interaction1.output_text)

In [ ]:
interaction2 = client.interactions.create(
    model="gemini-3.8-flash",
    input="How many paws are in my house?",
    previous_interaction_id=interaction1.id
)
print("Response 2:", interaction2.output_text)

Chat Conversation

In [ ]:
MODEL = "gemini-3-flash-preview"

In [ ]:
chat = client.chats.create(model=MODEL)   # apna verified naam

response = chat.send_message("In one sentence, explain how a computer works to a young child.")
display(to_markdown(response.text))

# Streaming ke saath dusra sawal
for chunk in chat.send_message_stream("Okay, how about a more detailed explanation to a high schooler?"):
    print(chunk.text)
    print("_" * 80)

# History dekhna
for message in chat.get_history():
    display(to_markdown(f"**{message.role}**: {message.parts[0].text}"))

Token Count


In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is the meaning of life?"
)
print(response.usage_metadata)

Use embeddings

In [ ]:
result = client.models.embed_content(
    model="gemini-embedding-001",   # list se verify karo
    contents="What is the meaning of life?",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        title="Embedding of single string",
    ),
)
vec = result.embeddings[0].values
print(len(vec), str(vec)[:50], "... TRIMMED")

# Strings ki list
texts = [
    "What is the meaning of life?",
    "How much wood would a woodchuck chuck?",
    "How does the brain work?",
]
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
for e in result.embeddings:
    print(str(e.values)[:50], "... TRIMMED")

Advanced Use Case - safety checking

In [ ]:


response = client.models.generate_content(
    model=MODEL,
    contents="What is sex education and why is it taught in schools?",
    config=types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                threshold=types.HarmBlockThreshold.BLOCK_ONLY_HIGH,
            ),
        ]
    ),
)


In [ ]:
if response.candidates:
    print(response.candidates[0].finish_reason)    # STOP matlab normal, SAFETY matlab block
    print(response.candidates[0].safety_ratings)
    print(response.text)
else:
    print("Block hua:", response.prompt_feedback)